# 04 — Confidence Classification (XGBoost 6-Class)

**Project:** MARS — Multi-Agent Recommender System for Personalized Learning  
**Agent:** ConfidenceAgent  
**Purpose:** Train and evaluate the 6-class confidence classifier,
analyse feature importance with SHAP, and generate paper figures.

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, f1_score,
)
from sklearn.preprocessing import label_binarize

plt.style.use("seaborn-v0_8-paper")
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300,
    "font.size": 11, "axes.titlesize": 13,
    "axes.labelsize": 12, "figure.figsize": (8, 5),
    "savefig.bbox": "tight",
})

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

import logging
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

print("Libraries loaded.")

## 1. Load Data & Train Classifier

In [ ]:
from data.loader import EdNetLoader
from agents.confidence_agent import (
    ConfidenceAgent, ConfidenceClass, CLASS_NAMES, SKILL_DELTAS, FEATURE_NAMES,
)
from agents.diagnostic_agent import DiagnosticAgent

loader = EdNetLoader(data_dir="../data/raw")
interactions = loader.load_interactions(sample_users=1000)
print(f"Interactions: {len(interactions):,} ({interactions['user_id'].nunique()} users)")

# Calibrate IRT for difficulty features
diag = DiagnosticAgent()
irt_params = diag.calibrate_from_interactions(interactions, min_answers_per_q=5)

In [ ]:
# Train confidence classifier
conf_agent = ConfidenceAgent()
metrics = conf_agent.train(interactions, irt_params=irt_params)

print("\nTraining Results:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

## 2. Class Distribution

In [ ]:
# Get labels for the dataset
df = interactions.dropna(subset=["elapsed_time", "correct", "changed_answer"]).copy()
labels = conf_agent._assign_labels(df)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of class distribution
counts = labels.value_counts().sort_index()
colors = ["#2ecc71", "#3498db", "#e74c3c", "#8e44ad", "#f39c12", "#e67e22"]
bars = axes[0].bar([CLASS_NAMES[i] for i in counts.index], counts.values,
                     color=colors, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + len(df)*0.005,
                 f"{val:,}\n({val/len(df)*100:.1f}%)", ha="center", fontsize=8)
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of 6 Confidence Classes")
axes[0].tick_params(axis="x", rotation=35)

# Skill deltas
delta_names = [CLASS_NAMES[i] for i in range(6)]
delta_vals = [SKILL_DELTAS[ConfidenceClass(i)] for i in range(6)]
bar_colors = ["#2ecc71" if d > 0 else "#e74c3c" for d in delta_vals]
axes[1].barh(delta_names, delta_vals, color=bar_colors, edgecolor="black", linewidth=0.5)
axes[1].set_xlabel("Skill Delta")
axes[1].set_title("Skill Update per Confidence Class")
axes[1].axvline(0, color="black", linewidth=0.5)

for ax in axes:
    sns.despine(ax=ax)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "fig_confidence_distribution.png")
plt.show()

## 3. Confusion Matrix

In [ ]:
# Predict on full dataset
X = conf_agent._build_features(df)
y_true = labels.values
y_pred = conf_agent.model.predict(X.values.astype(np.float32))

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            square=True, linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Normalised Confusion Matrix (6-Class Confidence)")

fig.savefig(RESULTS_DIR / "fig_confidence_confusion.png")
plt.show()

## 4. Classification Report

In [ ]:
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0, output_dict=True)
report_df = pd.DataFrame(report).T
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

# Bar chart of per-class F1
fig, ax = plt.subplots(figsize=(10, 5))
class_f1 = [report[c]["f1-score"] for c in CLASS_NAMES]
bars = ax.bar(CLASS_NAMES, class_f1, color=colors, edgecolor="black", linewidth=0.5)
for bar, val in zip(bars, class_f1):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f"{val:.3f}", ha="center", fontsize=9)
ax.axhline(report["macro avg"]["f1-score"], color="red", linestyle="--",
           label=f"Macro F1: {report['macro avg']['f1-score']:.3f}")
ax.set_ylabel("F1-Score")
ax.set_title("Per-Class F1-Score")
ax.set_ylim(0, 1.05)
ax.tick_params(axis="x", rotation=35)
ax.legend()
sns.despine()

fig.savefig(RESULTS_DIR / "fig_confidence_f1.png")
plt.show()

## 5. SHAP Summary Plot

In [ ]:
# SHAP analysis (subsample for speed)
X_sample = X.sample(min(2000, len(X)), random_state=42)
explainer = shap.TreeExplainer(conf_agent.model)
shap_values = explainer.shap_values(X_sample.values.astype(np.float32))

# shap_values is a list of 6 arrays (one per class) or a 3D array
if isinstance(shap_values, list):
    # Average absolute SHAP across classes for summary
    mean_abs_shap = np.mean([np.abs(sv) for sv in shap_values], axis=0)
else:
    mean_abs_shap = np.mean(np.abs(shap_values), axis=2)  # (samples, features)

fig, ax = plt.subplots(figsize=(10, 7))
feat_importance = pd.Series(mean_abs_shap.mean(axis=0), index=FEATURE_NAMES).sort_values()
feat_importance.plot.barh(ax=ax, color="#3498db", edgecolor="black", linewidth=0.3)
ax.set_xlabel("Mean |SHAP value| (avg across classes)")
ax.set_title("Feature Importance (SHAP)")
sns.despine()

fig.savefig(RESULTS_DIR / "fig_shap_summary.png")
plt.show()

## 6. SHAP Dependence Plots (Top-3 Features)

In [ ]:
top3 = feat_importance.tail(3).index.tolist()[::-1]

# Use class 0 (SOLID) SHAP values for dependence plots
if isinstance(shap_values, list):
    sv_class0 = shap_values[0]
else:
    sv_class0 = shap_values[:, :, 0]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, feat_name in enumerate(top3):
    feat_idx = FEATURE_NAMES.index(feat_name)
    ax = axes[i]
    ax.scatter(
        X_sample.values[:, feat_idx],
        sv_class0[:, feat_idx],
        c=X_sample.values[:, feat_idx],
        cmap="RdBu_r", s=8, alpha=0.5,
    )
    ax.set_xlabel(feat_name)
    ax.set_ylabel(f"SHAP value (class SOLID)")
    ax.set_title(f"SHAP Dependence: {feat_name}")
    ax.axhline(0, color="gray", linestyle=":", alpha=0.5)
    sns.despine(ax=ax)

fig.tight_layout()
fig.savefig(RESULTS_DIR / "fig_shap_dependence.png")
plt.show()

## 7. ROC Curves (Per-Class)

In [ ]:
# One-vs-rest ROC curves
y_proba = conf_agent.model.predict_proba(X.values.astype(np.float32))
y_bin = label_binarize(y_true, classes=list(range(6)))

fig, ax = plt.subplots(figsize=(9, 7))

for i in range(6):
    if y_bin[:, i].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[i], linewidth=1.5,
            label=f"{CLASS_NAMES[i]} (AUC={roc_auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=0.5, alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves per Confidence Class (One-vs-Rest)")
ax.legend(fontsize=9, loc="lower right")
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)
sns.despine()

fig.savefig(RESULTS_DIR / "fig_confidence_roc.png")
plt.show()

## 8. XGBoost Feature Importance (Gain)

In [ ]:
feat_imp = conf_agent.get_feature_importance()

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.sort_values().plot.barh(ax=ax, color="#27ae60", edgecolor="black", linewidth=0.3)
ax.set_xlabel("Feature Importance (Gain)")
ax.set_title("XGBoost Feature Importance")
sns.despine()

fig.savefig(RESULTS_DIR / "fig_xgb_importance.png")
plt.show()

In [ ]:
print("\n=== Confidence Classification Complete ===")
print(f"Model: XGBoost 6-class, {len(FEATURE_NAMES)} features")
print(f"CV F1-macro: {metrics['cv_f1_macro_mean']:.4f} +/- {metrics['cv_f1_macro_std']:.4f}")
print(f"Full F1-macro: {metrics['full_f1_macro']:.4f}")
print(f"SMOTE applied: {metrics['smote_applied']}")
print(f"Figures saved to: {RESULTS_DIR.resolve()}")